# 情感分析 NLP 方法 （一） —— 传统机器学习方法

## 导入第三方库

In [ ]:
# DATA
import numpy as np
import pandas as pd

# VIS
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# STD
import random
from time import time

# NLP
import nltk
nltk.download('stopwords')
nltk.download('punkt')
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
from nltk.stem.porter import PorterStemmer
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.lancaster import LancasterStemmer

# ML
from sklearn import preprocessing
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB,CategoricalNB
from sklearn.manifold import TSNE
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier

## 读取数据

In [ ]:
train = pd.read_csv('../input/nlp-getting-started/train.csv')
test = pd.read_csv('../input/nlp-getting-started/test.csv')

# EDA

In [ ]:
# 查看一下训练集和测试集的大小
print('train set:', train.shape,'\ntest set:' , test.shape)

In [ ]:
train.head(10)

In [ ]:
train.info()

In [ ]:
# 去除其中的重复值
train=train.drop_duplicates(subset=['text', 'target'], keep='first')
train.shape

可以看到有 92 个重复值.

In [ ]:
train.target.value_counts()

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(10,8),dpi=240)
train.groupby('target').id.count().plot.bar(ylim=0)
plt.show()

可以看到属于不平衡数据集，但是其实也没有太不平衡。

下面构造特征：单词个数

In [ ]:
train['text_length'] = train.text.apply(lambda x: len(x.split()))
test['text_length'] = test.text.apply(lambda x: len(x.split()))

In [ ]:
train['text_length'].describe()

In [ ]:
test['text_length'].describe()

* Max number of words in all data is 31 and min is 1!

In [ ]:
def plot_word_count(df, data_name):
  sns.distplot(df['text_length'].values)
  plt.title(f'Sequence char count: {data_name}')
  plt.grid(True)

In [ ]:
#ig = plt.figure(figsize=(16,6))
#plt.hist(train["text_length"], bins = 30)
#plt.show()
plt.subplot(1, 2, 1)
plot_word_count(train, 'Train')

plt.subplot(1, 2, 2)
plot_word_count(test, 'Test')

plt.subplots_adjust(right=3.0)
plt.show()

In [ ]:
# collecting all words in single list
list_= []
for i in train.text:
    list_ += i
list_= ''.join(list_)
allWords=list_.split()
vocabulary= set(allWords)

In [ ]:
len(vocabulary)

在训练数据里面有 31480 种不同的单词。

In [ ]:
def create_corpus(df,target):
    corpus=[]
    for x in df[df['target']==target]['text'].str.split():
        for i in x:
            corpus.append(i)
    return corpus

下面绘制图像来看训练集当中不同的标签对应的单词频次都如何？

In [ ]:
import collections
allWords=create_corpus(train,target=0)
vocabulary= set(allWords)
vocabulary_list= list(vocabulary)

plt.figure(figsize=(16,5), dpi=120)
counter=collections.Counter(allWords)
most=counter.most_common()
x=[]
y=[]
for word,count in most[:20]:
  x.append(word)
  y.append(count)
sns.barplot(x=y,y=x)

In [ ]:
#most frequent 20 words when label == 1 
import collections
allWords=create_corpus(train,target=1)
vocabulary= set(allWords)
vocabulary_list= list(vocabulary)

plt.figure(figsize=(16,5), dpi=120)
counter=collections.Counter(allWords)
most=counter.most_common()
x=[]
y=[]
for word,count in most[:20]:
  x.append(word)
  y.append(count)
sns.barplot(x=y,y=x)

# 文本清洗方式

## 去除标点符号

Python 官方给出了标点符号的使用库，里面内置了常用的标点符号.

In [ ]:
string.punctuation

In [ ]:
# 去除标点符号例子
text='hey # how are !you doing ?'
"".join([char for char in text if char not in string.punctuation])

## 2-Removing Numbers

In [ ]:
# 使用正则表达式去除数字
text='hey 4 look 333 at me0'
re.sub('[0-9]', '', text)

## 3-Removing Stopwords

停用词，即 the, a, for 这种定冠词、介词等没有意义的词组成的一类词汇，在实际使用当中会去掉.

In [ ]:
# 使用 NLTK 的停用词表
stopwords.words('english')[:8]

In [ ]:
# 使用例
text='hey this is me and I am here to help you  '
tokens = word_tokenize(text)
tokens=[word for word in tokens if word not in stopwords.words('english')]
' '.join(tokens)

## 集合上述的清洗方式

注意我们在使用的时候要把大写转换为小写，同时使用了词干提取器。

In [ ]:
stemmer = SnowballStemmer("english")

def clean_text(text):
    # 转小写
    text= text.lower()

    # 去掉数字
    text= re.sub('[0-9]', '', text)

    # 去掉标点符号
    text  = "".join([char for char in text if char not in string.punctuation])
    
    # 词干化
    tokens = word_tokenize(text)
    tokens=[stemmer.stem(word) for word in tokens]

    # 停用词
    tokens=[word for word in tokens if word not in stopwords.words('english')]
    text = ' '.join(tokens)
    
    return text

In [ ]:
clean_text("hey I am here # ! looks 4 GOOD can't see you!")

In [ ]:
train["clean"]=train["text"].apply(clean_text)
test["clean"]=test["text"].apply(clean_text)

In [ ]:
#Let's see the effect of cleaning
train[["text","clean"]].head(4)

In [ ]:
# collecting all words in single list
list_= []
for i in train.clean:
    list_ += i
list_= ''.join(list_)
allWords=list_.split()
vocabulary= set(allWords)
len(vocabulary)

* we reduced our data from 31480 unique words to 19920

In [ ]:
tfidf = TfidfVectorizer(sublinear_tf=True,max_features=60000, min_df=1, norm='l2',  ngram_range=(1,2))
features = tfidf.fit_transform(train.clean).toarray()
features.shape

In [ ]:
features_test = tfidf.transform(test.clean).toarray()

# 机器学习算法使用

In [ ]:
skf = StratifiedKFold(n_splits=4, random_state=42, shuffle=True)
y=train['target']

In [ ]:
X = features
X.shape

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
import lightgbm as lgb

def evaluate_models(X, y, n_splits=5, random_state=42):
    """
    使用K折交叉验证评估多种分类模型，并绘制F1分数条形图
    
    参数:
    X: 特征数据
    y: 标签数据
    n_splits: K折交叉验证的折数，默认为5
    random_state: 随机数种子，保证结果可复现
    """
    
    # 定义要评估的模型列表
    models = [
        ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=random_state)),
        ('Naive Bayes', GaussianNB()),
        ('Random Forest', RandomForestClassifier(random_state=random_state)),
        ('LDA', LinearDiscriminantAnalysis()),
        ('Gradient Boosting', GradientBoostingClassifier(random_state=random_state)),
        ('XGBoost', xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=random_state)),
        ('LGBM', lgb.LGBMClassifier(random_state=random_state)),
        ('kNN', KNeighborsClassifier())
    ]
    
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    f1_scorer = make_scorer(f1_score)
    results = []
    names = []
    mean_scores = []
    
    
    for name, model in models:
        print(f'[INFO] 正在进行模型 {name} 的训练；')
        cv_results = cross_val_score(model, X, y, cv=kfold, scoring=f1_scorer)
        results.append(cv_results)
        names.append(name)
        mean_score = cv_results.mean()
        mean_scores.append(mean_score)
        print("模型交叉验证F1分数：")
        print(f"{name}: 均值 = {mean_score:.4f}, 标准差 = {cv_results.std():.4f}")
    
    sorted_indices = np.argsort(mean_scores)[::-1]
    sorted_names = [names[i] for i in sorted_indices]
    sorted_scores = [mean_scores[i] for i in sorted_indices]
    
    plt.figure(figsize=(12, 8), dpi=240)
    bars = plt.bar(sorted_names, sorted_scores, color='skyblue')
    
    plt.ylim(0, 1.0)  # F1分数范围在0-1之间
    plt.xlabel('Classification Model', fontsize=12)
    plt.ylabel('F1 Score', fontsize=12)
    plt.title(f'{n_splits}-Fold Validation Scores By Models', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.4f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()

    # 找到表现最佳的模型
    best_idx = np.argmax(mean_scores)
    best_name = names[best_idx]
    best_model = models[best_idx][1]
    
    return best_model, best_name

In [ ]:
best_model, best_name = evaluate_models(X, y, n_splits=5)
print('最优模型：',best_name)

# Submission

In [ ]:
best_model.fit(X, y)
answer = best_model.predict(features_test)

In [ ]:
submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
submission['target'] = answer

In [ ]:
submission.to_csv('submission.csv',index=False)